# Linux Wildcards and the find Command (Educational Notebook)
This notebook covers shell wildcard (glob) expansion for matching multiple filenames at once, and the `find` command for locating files by more complex, arbitrary criteria.

## 1. Why Wildcards?

When you type a command containing special pattern characters like `*`, the **shell itself** (not the command you're running) expands that pattern into a list of matching filenames *before* the command ever sees it. This is called **globbing**.

```bash
echo *.txt         # the shell replaces *.txt with every matching filename in the current directory
ls *.txt              # ls never sees "*.txt" -- it only sees the already-expanded list of real filenames
```

If no filename matches the pattern, most shells (including Bash by default) pass the literal, unexpanded pattern string through to the command instead - which is why an unmatched wildcard sometimes produces a confusing "no such file" error naming the pattern itself.


## 2. The `*` Wildcard

`*` matches **any sequence of characters, including none at all**, within a single path component (it does not cross a `/` into a subdirectory on its own).

```bash
ls *.txt          # every file ending in .txt in the current directory
ls report*           # every file starting with "report"
ls *.log*              # every file with .log anywhere in the name
```

By default, `*` does **not** match filenames that start with a dot (hidden files) - `ls *` will skip `.bashrc`, for example. This is a deliberate safety measure so that a broad wildcard doesn't casually sweep up hidden configuration files.


## 3. The `?` Wildcard

`?` matches **exactly one** character, no more and no less.

```bash
ls file?.txt          # matches file1.txt, file2.txt, fileA.txt -- but not file10.txt or file.txt
ls ??.txt                # matches any two-character filename ending in .txt, e.g. ab.txt
```


## 4. Character Classes and Ranges

Square brackets match **exactly one character**, chosen from a specified set or range.

```bash
ls file[123].txt        # matches file1.txt, file2.txt, file3.txt only
ls file[1-3].txt           # same thing, expressed as a range
ls file[a-z].txt             # a single lowercase letter
ls file[!0-9].txt              # negation: one character that is NOT a digit (some shells also accept [^0-9])
```


## 5. Brace Expansion

**Brace expansion** (`{...}`) is technically a separate Bash feature from globbing - it expands *unconditionally*, purely as text substitution, whether or not any matching files exist.

```bash
echo file{1,2,3}.txt          # expands to: file1.txt file2.txt file3.txt (no files need to exist)
mkdir -p project/{src,docs,tests}    # a common trick: create several sibling directories in one command
echo {1..5}                            # numeric range: 1 2 3 4 5
echo {a..e}                              # letter range: a b c d e
cp important.conf important.conf.bak      # the manual equivalent of...
cp important.conf{,.bak}                    # ...this brace-expansion shorthand
```

Because brace expansion doesn't check the filesystem at all, `echo {1..3}.txt` will happily print `1.txt 2.txt 3.txt` even if none of those files exist - unlike `*` or `?`, which only ever expand to files that are actually there.


## 6. Hidden Files and `dotglob`

As noted above, `*` skips dotfiles by default. To match them explicitly:

```bash
ls .*             # matches only names starting with a dot (includes . and .. themselves)
ls -A                # list almost all entries, including dotfiles, but excluding . and ..
shopt -s dotglob       # (Bash) make * itself match dotfiles too, for the rest of the session
```


## 7. Introducing `find`

Wildcards are expanded by the shell and only look at filenames in the directories you name. `find` is a full command in its own right that walks a directory tree recursively and selects files based on much richer criteria - name, type, size, age, permissions, and more.

```bash
find /var/log                         # list every file and directory under /var/log, recursively
find . -name "*.txt"                     # find files matching a name pattern, starting from the current directory
```

General syntax: `find [starting-path...] [expression]`, where the expression is a chain of tests and actions.


## 8. Common `find` Tests

```bash
find . -name "*.conf"           # match by name (case-sensitive)
find . -iname "*.conf"            # match by name, case-insensitive

find . -type f                      # regular files only
find . -type d                        # directories only
find . -type l                          # symbolic links only

find . -size +10M                         # files larger than 10 MB
find . -size -1k                            # files smaller than 1 KB

find . -mtime -7                              # modified within the last 7 days
find . -mtime +30                               # modified more than 30 days ago
find . -newer reference.txt                       # modified more recently than a reference file

find / -user alice                                  # owned by a specific user
find / -perm 777                                       # an exact permission mode
```

Multiple tests combine with an implicit **AND**: `find . -name "*.log" -mtime +30` matches files that satisfy *both* conditions.


## 9. Acting on Results

By default `find` just prints matching paths, but it can also act on each match directly:

```bash
find . -name "*.tmp" -delete                        # delete every match directly -- powerful and dangerous, test with a plain find first
find . -name "*.log" -exec gzip {} \;                  # run a command on each match; {} is replaced with the matched path, \; ends the command
find . -name "*.log" -exec rm {} +                        # a faster variant: batches many matches into fewer invocations of the command

find . -name "*.bak" | xargs rm                              # pipe matches into xargs, which builds and runs a command from its input
find . -name "*.bak" -print0 | xargs -0 rm                     # the -print0 / -0 pairing safely handles filenames containing spaces or newlines
```

Always run a plain `find` (without `-delete` or `-exec rm`) first to confirm exactly which files match, before adding a destructive action.


## 10. Wildcards vs. `find`: Choosing the Right Tool

| | Wildcards (`*`, `?`, `[...]`) | `find` |
|---|---|---|
| Expanded by | The shell, before the command runs | `find` itself, as it walks the tree |
| Searches | Only the directory (or directories) you name | Recursively, through an entire tree by default |
| Criteria | Filename pattern only | Name, type, size, age, owner, permissions, and more, combinable |
| Can take action per match | No (the matched command decides what to do with all of them at once) | Yes, directly (`-delete`, `-exec`) |

Use a wildcard for a quick, simple, one-directory filename pattern; reach for `find` as soon as you need recursion, non-name criteria, or per-file actions.


## Hands-on

Try these on your own system:

```bash
mkdir -p ~/findtest/{a,b} && cd ~/findtest
touch a/one.txt a/two.log b/three.txt notes.md .hidden

ls *.txt
ls *
ls -A

find . -name "*.txt"
find . -type f
find . -type d
find . -name "*.log" -exec echo "would gzip: {}" \;
```

Compare what each `ls` wildcard picks up versus what `find` finds recursively.


## Review Questions

1. Why does `*.txt` sometimes cause a "no such file" error naming the literal pattern, instead of matching nothing silently?
2. What's the difference between what `*` matches and what `?` matches?
3. Why doesn't `*` match hidden (dotfile) filenames by default, and how would you make it do so?
4. What is the key difference between brace expansion (`{a,b,c}`) and globbing (`*`, `?`, `[...]`) in terms of whether matching files need to exist?
5. Write a `find` command that lists every `.log` file under `/var/log` larger than 5 MB.
6. Why should you run a plain `find` before adding `-delete` to it?
7. What does `{}` mean inside a `find -exec` command, and what does `\;` do?
8. When would you reach for `find` instead of a shell wildcard?


# Cheat Sheet

```
Wildcards (expanded by the shell):
  *          any characters (not across a leading dot)
  ?            exactly one character
  [abc] [a-z]    one character from a set/range
  [!abc]            negated set
  {a,b,c}             brace expansion (unconditional, no filesystem check)
  {1..5}                numeric/letter range

Hidden files:
  ls -A        shopt -s dotglob

find basics:
  find <path> -name "pattern"      find <path> -iname "pattern"  (case-insensitive)
  find <path> -type f|d|l
  find <path> -size +10M | -1k
  find <path> -mtime -7 | +30
  find <path> -newer ref.txt

find actions:
  find ... -delete
  find ... -exec cmd {} \;         one invocation per match
  find ... -exec cmd {} +            batched invocations
  find ... -print0 | xargs -0 cmd      safe for filenames with spaces
```
